# Single

In [ ]:
%matplotlib inline
from pathlib import Path

from susi.io.app_settings import AppSettings
import susi.io.load_output_data as load_output
from analysis.notebooks.components import (
    folder_selection,
    metadata_display,
    variable_selection,
    quick_look_plots,
)

## Choose project folder

Browse to a custom folder below, or skip this cell to use the default data folder (`AppSettings().output_folder`) in the next one.

In [ ]:
folder_browser = folder_selection.build_folder_browser(AppSettings().output_folder)

In [ ]:
data_folder = folder_selection.resolve_start_folder(
    folder_browser, default=AppSettings().output_folder
)
project_dropdown = folder_selection.build_dropdown(data_folder, label="project")
project_dropdown

In [ ]:
stand_dropdown = folder_selection.build_dropdown(project_dropdown.value, label="stand")
stand_dropdown

In [ ]:
scenario_dropdown = folder_selection.build_dropdown(stand_dropdown.value, label="scenario")
scenario_dropdown

In [ ]:
chosen_scenario_folder = scenario_dropdown.value

## Metadata

In [ ]:
params = load_output.read_params_from_jsons(
    experiment_folderpath=chosen_scenario_folder
)
metadata_display.display_metadata(params.metadata, params.susi_params)

## Choose variables

In [ ]:
# The golden test netcdf is used only to read the variable structure of the
# netcdf file (names/shapes/units), not its values.
sample_netcdf_filepath = (
    AppSettings().project_root_path / "tests/golden_file_test/golden_susi.nc"
)
all_variables = load_output.list_all_netcdf_variables(sample_netcdf_filepath)

selector = variable_selection.build_selector(all_variables)

## Plots

In [ ]:
chosen_netcdf_variables = variable_selection.checked_variables(selector, all_variables)

if not chosen_netcdf_variables:
    print("No variables chosen")
else:
    chosen_netcdf_filepath = Path(params.metadata["netcdf_output_filepath"])
    variables_values = load_output.read_value_several_variables_from_single_file(
        netcdf_filepath=chosen_netcdf_filepath,
        variable_paths=list(chosen_netcdf_variables.keys()),
    )
    quick_look_plots.display_quick_look_plots(variables_values)